# Delta Lake smoke test on MinIO

Open in VS Code with your repo's Python kernel. Install `pyspark==3.5.6` and `python-dotenv` in that kernel if needed, then restart it. Use JDK 17 for Spark 3.5.6.

Create `data-platform/.env` with `AWS_ACCESS_KEY_ID` and `AWS_SECRET_ACCESS_KEY`. Keep it ignored by Git. Open the repository in VS Code and run all cells. The `bronze` bucket must exist. Each run uses a new path under `bronze/_smoke/`.

In [ ]:
import os
import sys
from pathlib import Path
from uuid import uuid4

from dotenv import dotenv_values
from pyspark.sql import SparkSession

cwd = Path.cwd().resolve()
env_file = next(
    (folder / ".env" for folder in (cwd, *cwd.parents)
     if folder.name == "data-platform"),
    cwd / "data-platform" / ".env",
)
if not env_file.is_file():
    raise FileNotFoundError(f"Create {env_file} and open the repo in VS Code.")

credentials = dotenv_values(env_file, interpolate=False)
required = ("AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY")
missing = [name for name in required if not credentials.get(name)]
if missing:
    raise ValueError(f"Missing credentials in {env_file}: {', '.join(missing)}")

for name in required:
    os.environ[name] = credentials[name]
del credentials

os.environ["PYSPARK_PYTHON"] = sys.executable

spark = (
    SparkSession.builder.master("local[2]")
    .appName("reconciliation-delta-smoke-test")
    .config("spark.jars.packages",
            "io.delta:delta-spark_2.12:3.3.3,org.apache.hadoop:hadoop-aws:3.3.4")
    .config("spark.jars.ivy", str(Path.home() / ".ivy2"))
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.shuffle.partitions", "2")
    .config("spark.databricks.delta.snapshotPartitions", "2")
    .config("spark.hadoop.fs.s3a.endpoint", "http://192.168.29.250:9000")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "com.amazonaws.auth.EnvironmentVariableCredentialsProvider")
    .getOrCreate()
)

In [ ]:
table_path = f"s3a://bronze/_smoke/delta-{uuid4().hex}"
expected = [(1, "write"), (2, "read")]

# Generate the test rows in Spark SQL.
source = spark.sql("SELECT * FROM VALUES (1, 'write'), (2, 'read') AS data(id, label)")
source.write.format("delta").mode("errorifexists").save(table_path)

result = spark.read.format("delta").load(table_path).orderBy("id")
actual = [(row.id, row.label) for row in result.collect()]
if actual != expected:
    raise RuntimeError(f"Expected {expected}, got {actual}")

result.show()
print(f"PASS: wrote and read {len(actual)} Delta rows at {table_path}")